# CSIRO Competition Solution Notebook

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID1717006117.jpg
/kaggle/input/csiro-biomass/train/ID1638922597.jpg
/kaggle/input/csiro-biomass/train/ID475010202.jpg
/kaggle/input/csiro-biomass/train/ID1857489997.jpg
/kaggle/input/csiro-biomass/train/ID684383343.jpg
/kaggle/input/csiro-biomass/train/ID605134229.jpg
/kaggle/input/csiro-biomass/train/ID1463690813.jpg
/kaggle/input/csiro-biomass/train/ID1403078396.jpg
/kaggle/input/csiro-biomass/train/ID1997244125.jpg
/kaggle/input/csiro-biomass/train/ID545360459.jpg
/kaggle/input/csiro-biomass/train/ID1783499590.jpg
/kaggle/input/csiro-biomass/train/ID157479394.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID839432753.jpg
/kaggle/input/csiro-biomass/train/ID2030696575.jpg
/kaggle/input/csiro-biomass/train/ID710341728.jpg
/kaggle/input/cs

In [3]:
import torch

torch.cuda.is_available()

True

# Data Prep

## Data Augmentation & Transform

In [4]:
# Data Transform

from torchvision.transforms import v2
import torch

# to_tensor = v2.ToTensor()
# img_tensor = to_tensor(img)

dtype = torch.float32
img_size = (224, 224)
from torchvision.transforms import v2
import torch

dtype = torch.float32
img_size = (224, 224)

# Aggressive training transform for small dataset
train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),
    v2.Resize((256, 256)),  # Larger for cropping
    
    # Geometric augmentations
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(180, interpolation=v2.InterpolationMode.BILINEAR),
    v2.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.85, 1.15),
        shear=10,
        interpolation=v2.InterpolationMode.BILINEAR,
    ),
    v2.RandomResizedCrop(
        size=img_size,
        scale=(0.75, 1.0),
        ratio=(0.9, 1.1),
        interpolation=v2.InterpolationMode.BILINEAR,
    ),
    
    # Color augmentations
    v2.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.1),
    v2.RandomApply([v2.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),
    v2.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    v2.RandomAutocontrast(p=0.2),
    v2.RandomGrayscale(p=0.05),
    v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )

])

# Clean validation transform
val_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype, scale=True),
    v2.Resize(img_size),
    v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
)
])




def numeric_transform(X, X_max, X_min) -> torch.Tensor:
    X_normalized = (X - X_min) / (X_max - X_min)
    return X_normalized

def target_transform(targets) -> torch.Tensor:
    return torch.log1p(targets)

def target_untransform(targets) -> torch.Tensor:
    return torch.expm1(targets)

def categorical_transform(row) -> torch.Tensor:
    return row

## Train Set

In [5]:

# from torch.utils.data import Dataset
# from torchvision.io import decode_image
# from sklearn.preprocessing import LabelEncoder
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader
# import pandas as pd

# class Image2BioMassTrainValDataset(Dataset):
    
#     def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None, target_transform=None):
        
#         self.df = self.process_df(dataset_path)
#         self.dataset_path = dataset_path
#         self.img_transform = img_transform
#         self.target_transform = target_transform
#         self.numeric_transform = numeric_transform
#         self.categorical_transform = categorical_transform
#         self.targets = self.df.loc[:, ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]]


#     def process_df(self, dataset_path):
#         self.le_date = LabelEncoder()
#         self.le_state = LabelEncoder()
#         self.le_species = LabelEncoder()

#         df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
#         df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
#         df = df.pivot_table(
#         index=['base_sample_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
#         columns='target_name',
#         values='target'
#         ).reset_index()
#         df["Sampling_Date"] = self.le_date.fit_transform(df["Sampling_Date"])
#         df["State"] = self.le_state.fit_transform(df["State"])
#         df["Species"] = self.le_species.fit_transform(df["Species"])
#         # display(df)
#         return df

#     def __len__(self):
#         return len(self.df)

#     def get_cat_features(self):
#         return ["Sampling_Date", "State", "Species"]
    
#     def get_cat_vocab_sizes(self):
#         results = []

#         for i in self.get_cat_features():
#             results.append(len(self.df[i].unique()))
#         return results

#     def __getitem__(self, idx):
#         # B = batch_size
#         # display(self.df)
#         img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
#         image = decode_image(img_path)
#         # display(self.df)
#         numeric_features = torch.tensor([
#             self.df.loc[idx, "Pre_GSHH_NDVI"],
#             self.df.loc[idx, "Height_Ave_cm"],
#         ], dtype=torch.float32)

#         categorical_features = torch.tensor([
#             self.df.loc[idx, "Sampling_Date"],
#             self.df.loc[idx, "State"],
#             self.df.loc[idx, "Species"],
#         ], dtype=torch.long)
        

#         if self.img_transform:
#             image = self.img_transform(image)
            
#         if self.numeric_transform:
#             # numeric_features[0] = self.numeric_transform(
#             #     numeric_features[0],
#             #     self.df.loc[:, "Pre_GSHH_NDVI"].max(), 
#             #     self.df.loc[:, "Pre_GSHH_NDVI"].min()
#             # )
#             numeric_features[1] = self.numeric_transform(
#                 numeric_features[1], 
#                 self.df.loc[:, "Height_Ave_cm"].max(), 
#                 self.df.loc[:, "Height_Ave_cm"].min()
#             )
#             # print(numeric_features)
#         combined_features = torch.cat([categorical_features.float(), numeric_features], dim=0)
#         # print(combined_features)
#         targets = torch.Tensor(self.targets.iloc[idx].values)
#         if self.target_transform:
#             targets = self.target_transform(targets)
#         return image, combined_features, targets

import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np


class Image2BioMassTrainValDataset(Dataset):
    """
    Robust dataset class with proper indexing and transform handling.
    """
    
    def __init__(
        self, 
        dataset_path, 
        indices=None,
        img_transform=None, 
        numeric_transform=None,
        target_transform=None,
        df=None,
        label_encoders=None,
        numeric_stats=None,
    ):
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.target_transform = target_transform
        self.numeric_transform = numeric_transform
        
        # If shared df and encoders provided, use them (for val set)
        if df is not None:
            self.df = df
            self.le_date = label_encoders['date']
            self.le_state = label_encoders['state']
            self.le_species = label_encoders['species']
            self.numeric_stats = numeric_stats
        else:
            self.df, self.le_date, self.le_state, self.le_species, self.numeric_stats = self._process_df()
        
        # Use subset of indices if provided
        if indices is not None:
            self.df = self.df.iloc[indices].reset_index(drop=True)
        
        self.targets = self.df[["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]].values

    def _process_df(self):
        le_date = LabelEncoder()
        le_state = LabelEncoder()
        le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(self.dataset_path, "train.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = df.pivot_table(
            index=['base_sample_id', 'image_path', 'Sampling_Date', 'State', 
                   'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm'],
            columns='target_name',
            values='target'
        ).reset_index()
        
        df["Sampling_Date"] = le_date.fit_transform(df["Sampling_Date"])
        df["State"] = le_state.fit_transform(df["State"])
        df["Species"] = le_species.fit_transform(df["Species"])
        
        # Store numeric stats for normalization
        numeric_stats = {
            'Pre_GSHH_NDVI': {'min': df['Pre_GSHH_NDVI'].min(), 'max': df['Pre_GSHH_NDVI'].max()},
            'Height_Ave_cm': {'min': df['Height_Ave_cm'].min(), 'max': df['Height_Ave_cm'].max()},
        }
        
        return df, le_date, le_state, le_species, numeric_stats

    def __len__(self):
        return len(self.df)

    def get_label_encoders(self):
        return {
            'date': self.le_date,
            'state': self.le_state,
            'species': self.le_species,
        }
    
    def get_cat_vocab_sizes(self):
        return [
            len(self.le_date.classes_),
            len(self.le_state.classes_),
            len(self.le_species.classes_),
        ]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = os.path.join(self.dataset_path, row['image_path'])
        image = decode_image(img_path)
        
        # Numeric features
        ndvi = row["Pre_GSHH_NDVI"]
        height = row["Height_Ave_cm"]
        
        # Normalize numeric features
        if self.numeric_transform:
            height = self.numeric_transform(
                height,
                self.numeric_stats['Height_Ave_cm']['max'],
                self.numeric_stats['Height_Ave_cm']['min']
            )
        
        numeric_features = torch.tensor([ndvi, height], dtype=torch.float32)

        # Categorical features
        categorical_features = torch.tensor([
            row["Sampling_Date"],
            row["State"],
            row["Species"],
        ], dtype=torch.long)

        # Apply image transform
        if self.img_transform:
            image = self.img_transform(image)

        # Combine features
        combined_features = torch.cat([categorical_features.float(), numeric_features], dim=0)
        
        # Targets
        targets = torch.tensor(self.targets[idx], dtype=torch.float32)
        if self.target_transform:
            targets = self.target_transform(targets)
            
        return image, combined_features, targets


## Test Set

In [6]:

from torch.utils.data import Dataset
from torchvision.io import decode_image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

class Image2BioMassTestDataset(Dataset):
    
    def __init__(self, dataset_path, img_transform=None, numeric_transform=None,categorical_transform=None):
        
        self.df = self.process_df(dataset_path)
        self.dataset_path = dataset_path
        self.img_transform = img_transform
        self.numeric_transform = numeric_transform
        self.categorical_transform = categorical_transform

    def process_df(self, dataset_path):
        self.le_date = LabelEncoder()
        self.le_state = LabelEncoder()
        self.le_species = LabelEncoder()

        df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
        df['base_sample_id'] = df['sample_id'].str.split('__').str[0]
        df = (
            df.assign(_val="")
              .pivot(index=['base_sample_id', "image_path"],
                     columns='target_name',
                     values='_val')
              .reset_index()
        )

        return df

    def __len__(self):
        return len(self.df)

    def get_cat_features(self):
        return ["Sampling_Date", "State", "Species"]
    
    def get_cat_vocab_sizes(self):
        results = []

        for i in self.get_cat_features():
            results.append(len(self.df[i].unique()))
        return results

    def __getitem__(self, idx):

        img_path = os.path.join(self.dataset_path, self.df.loc[idx, 'image_path'])
        image = decode_image(img_path)

        # Use val_transform for test data (no augmentation)
        if self.img_transform:
            image = self.img_transform(image)
        else:
            # Fallback basic transform if no transform provided
            transform = v2.Compose([
                v2.ToImage(),
                v2.ToDtype(dtype, scale=True),
                v2.Resize((518, 518)),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            image = transform(image)

        combined_features = torch.zeros(5, dtype=torch.float32)
        sample_id = self.df.loc[idx, 'base_sample_id']
        return image, combined_features, sample_id

In [7]:
test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=val_transform,  # Use val_transform (no augmentation, proper size)
    
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
next(iter(test_dataloader))[2]

('ID1001187975',)

## Train Split

In [8]:
import random
import numpy as np
from sklearn.model_selection import KFold

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Create base dataset to get full DataFrame and encoders
base_dataset = Image2BioMassTrainValDataset(
    dataset_path="/kaggle/working/csiro-biomass/",
    img_transform=None,
    numeric_transform=None,
    target_transform=None,
)

# Setup K-Fold Cross Validation
N_FOLDS = 5
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f"Total samples: {len(base_dataset)}")
print(f"K-Fold Cross Validation with {N_FOLDS} folds")
print(f"Each fold will have ~{len(base_dataset)//N_FOLDS} validation samples")

fold_splits = list(kfold.split(range(len(base_dataset))))

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    assert len(set(train_idx) & set(val_idx)) == 0, f"Data leakage in fold {fold_idx}!"
    print(f"Fold {fold_idx+1}: Train={len(train_idx)}, Val={len(val_idx)}")

Total samples: 356
K-Fold Cross Validation with 5 folds
Each fold will have ~71 validation samples
Fold 1: Train=284, Val=72
Fold 2: Train=285, Val=71
Fold 3: Train=285, Val=71
Fold 4: Train=285, Val=71
Fold 5: Train=285, Val=71


# Model

In [9]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models import resnet152, ResNet152_Weights
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel


class BackBone(nn.Module):

    def __init__(self):

        
        super().__init__()
        pass

    def forward(self, x):
        pass

class Image2BiomassModel(nn.Module):

    def __init__(self):
        super().__init__()

        # ---- load DINOv3 ConvNeXt Large backbone from local ----
        # Path to local DINOv3 ConvNeXt Large weights
        model_path = "/mnt/d/Sayid/Projects/Image2Biomass/CSIRO-Image2Biomass-Prediction/pretrained/dinov3-convnext-large/weights"
        # model_path = "facebook/dinov3-convnext-large-pretrain-lvd1689m"

        self.backbone = AutoModel.from_pretrained(
            model_path,
            trust_remote_code=False
        )
        
        # Freeze backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # DINOv3 ConvNeXt Large outputs 1536-dim features (from the pooler)
        self.fc1 = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.BatchNorm1d(1024),
            nn.Mish(),
            nn.Dropout(0.1),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.Mish(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Mish(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.Mish(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.Mish(),
            nn.Dropout(0.1)
        )

        # self.fc2 = nn.Sequential(
        #     nn.Linear(1024, 512),
        #     nn.LayerNorm(512),
        #     nn.GeLU(),
        #     nn.Dropout(0.4),
        #     nn.Linear(512, 512),
        #     nn.LayerNorm(512),
        #     nn.Mish(),
        #     nn.Linear(512, 512),
        #     nn.LayerNorm(512),
        # )

        # self.residual = nn.Sequential(
        #     nn.Linear(512, 512),
        #     nn.LayerNorm(512),
        #     nn.Mish(),
        #     nn.Linear(512, 512),
        #     nn.LayerNorm(512),
        # )

        self.out = nn.Linear(64, 3)

        self.criterion = nn.SmoothL1Loss(reduction="mean")

    def forward(self, x, y=None):
        # DINOv3 ConvNeXt expects normalized images and outputs pooled features
        outputs = self.backbone(x)
        # Use the pooler_output which is the global representation (1536-dim for ConvNeXt Large)
        x = outputs.pooler_output
        
        x = self.fc1(x)
        
        preds = self.out(x)

        loss = None
        if y is not None:
            loss = self.criterion(preds, y)

        return preds, loss



# sample = next(iter(train_dataloader))
# model = Image2BiomassModel()

# model(sample[0], sample[2])

In [10]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Train Loop

In [11]:
import os

# Create results directory if it doesn't exist
os.makedirs("train_results", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
BATCH_SIZE = 8
EPOCHS = 1000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2

# Create generator for reproducibility
g = torch.Generator()
g.manual_seed(42)

# Weights for R2 calculation
weights = torch.tensor([0.1, 0.1, 0.1, 0.2, 0.5], device=device)

Using device: cuda


In [12]:
def weighted_r2(y_true, y_pred, weights):
    y_true = target_untransform(y_true)
    y_pred = target_untransform(y_pred)

    
    # create new columns
    gdm = (y_true[:, 0] + y_true[:, 2]).unsqueeze(1)   # (batch, 1)
    tot = (y_true[:, 0] + y_true[:, 1] + y_true[:, 2]).unsqueeze(1)
    
    gdm_pred = (y_pred[:, 0] + y_pred[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_pred = (y_pred[:, 0] + y_pred[:, 1] + y_pred[:, 2]).unsqueeze(1)

    # append columns
    y_true = torch.cat([y_true, gdm, tot], dim=1)
    y_pred = torch.cat([y_pred, gdm_pred, tot_pred], dim=1)

    # print("Prediction:", y_pred)
    # print("Target:", y_true)

    # compute weighted R2
    mean = y_true.mean(dim=0)
    SSE = ((y_true - y_pred)**2).sum(dim=0)
    TSS = ((y_true - mean)**2).sum(dim=0)
    TSS = torch.clamp(TSS, min=1e-8)
    R2 = 1 - SSE / TSS
    R2 = torch.clamp(R2, min=-10, max=1)
    return (R2 * weights).sum() / weights.sum()


def weighted_r2_single(y_true, y_pred):
    """
    Compute R2 for each individual target separately.
    Returns dict with R2 for each target:
    - Dry_Green_g (y[0])
    - Dry_Dead_g (y[1])
    - Dry_Clover_g (y[2])
    - GDM_g (y[0] + y[2])
    - Dry_Total_g (y[0] + y[1] + y[2])
    """
    y_true = target_untransform(y_true)
    y_pred = target_untransform(y_pred)
    
    # create new columns for GDM and Total
    gdm_true = (y_true[:, 0] + y_true[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_true = (y_true[:, 0] + y_true[:, 1] + y_true[:, 2]).unsqueeze(1)
    
    gdm_pred = (y_pred[:, 0] + y_pred[:, 2]).unsqueeze(1)   # (batch, 1)
    tot_pred = (y_pred[:, 0] + y_pred[:, 1] + y_pred[:, 2]).unsqueeze(1)
    
    # append columns
    y_true_full = torch.cat([y_true, gdm_true, tot_true], dim=1)
    y_pred_full = torch.cat([y_pred, gdm_pred, tot_pred], dim=1)
    
    # compute R2 for each target separately
    mean = y_true_full.mean(dim=0)  # (5,)
    SSE = ((y_true_full - y_pred_full)**2).sum(dim=0)  # (5,)
    TSS = ((y_true_full - mean)**2).sum(dim=0)  # (5,)
    TSS = torch.clamp(TSS, min=1e-8)
    R2 = 1 - SSE / TSS  # (5,)
    R2 = torch.clamp(R2, min=-10, max=1)
    
    target_labels = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    
    return {label: r2_val.item() for label, r2_val in zip(target_labels, R2)}

In [13]:
import wandb
import os
os.environ["WANDB_API_KEY"] = "f5498d8776689da0795dbdee5044ad07e5c956ad"
wandb.login(key=os.environ["WANDB_API_KEY"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ser/.netrc
wandb: Currently logged in as: sayid-10121012 (sayid-10121012-universitas-komputer-indonesia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [14]:
from tqdm import tqdm
import torch
from torch.nn.utils import clip_grad_norm_
import os
import time

# Store results across all folds for comparison
all_folds_results = []
BATCH_SIZE=8


# K-Fold Cross Validation Training Loop
for fold_idx, (train_indices, val_indices) in enumerate(fold_splits):
    print(f"\n{'='*80}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*80}")
    
    # Create fold directory
    fold_dir = f"train_results/fold{fold_idx+1}"
    os.makedirs(fold_dir, exist_ok=True)
    
    # Reset seed for each fold
    set_seed(42 + fold_idx)
    
    # Create datasets for this fold with proper transforms
    train_dataset_fold = Image2BioMassTrainValDataset(
        dataset_path="/kaggle/working/csiro-biomass/",
        indices=train_indices,
        img_transform=train_transform,  # WITH augmentation
        numeric_transform=numeric_transform,
        target_transform=target_transform,
        df=base_dataset.df.copy(),
        label_encoders=base_dataset.get_label_encoders(),
        numeric_stats=base_dataset.numeric_stats,
    )
    
    val_dataset_fold = Image2BioMassTrainValDataset(
        dataset_path="/kaggle/working/csiro-biomass/",
        indices=val_indices,
        img_transform=val_transform,  # WITHOUT augmentation
        numeric_transform=numeric_transform,
        target_transform=target_transform,
        df=base_dataset.df.copy(),
        label_encoders=base_dataset.get_label_encoders(),
        numeric_stats=base_dataset.numeric_stats,
    )
    
    # Create dataloaders for this fold
    train_dataloader = DataLoader(
        train_dataset_fold,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )
    
    val_dataloader = DataLoader(
        val_dataset_fold,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    
    print(f"Train batches: {len(train_dataloader)}")
    print(f"Val batches: {len(val_dataloader)}")
    
    # Initialize NEW model for this fold (critical for K-Fold!)
    model = Image2BiomassModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    
    # Initialize W&B for this fold with enhanced configuration
    run = wandb.init(
        project="IMAGE2BIOMASSPREDICTION",
        name=f"fold{fold_idx+1}_dinov3-convnext-large",
        group="5fold-cv-dinov3",  # Group all folds together for comparison
        job_type=f"fold{fold_idx+1}",
        tags=["dinov3", "convnext-large", "5fold-cv", f"fold{fold_idx+1}"],
        config={
            "architecture": "dinov3-convnext-large",
            "backbone": "DINOv3 ConvNeXt Large (198M params)",
            "dataset": "Image2Biomass",
            "epochs": EPOCHS,
            "fold": fold_idx + 1,
            "n_folds": N_FOLDS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "AdamW",
            "loss_function": "SmoothL1Loss (beta=0.5)",
            "l1_regularization": 1e-7,
            "gradient_clip": 1.0,
            "train_samples": len(train_dataset_fold),
            "val_samples": len(val_dataset_fold),
            "augmentation": "aggressive",
            "image_size": img_size,
        },
        reinit=True,
    )
    
    # Log model architecture
    wandb.watch(model, log="all", log_freq=100, log_graph=True)
    
    # Training tracking
    best_val_r2 = -float('inf')
    best_epoch = 0
    best_val_r2_individual = None  # Track best R2 per target at best epoch
    train_losses, val_losses = [], []
    train_r2_history, val_r2_history = [], []
    fold_start_time = time.time()
    
    # Training loop for this fold
    for epoch in range(1, EPOCHS + 1):
        epoch_start_time = time.time()
        
        model.train()
        train_loss = 0
        train_r2_scores = []
        train_r2_individual = {
            "Dry_Green_g": [],
            "Dry_Dead_g": [],
            "Dry_Clover_g": [],
            "GDM_g": [],
            "Dry_Total_g": []
        }
        
        for imgs, _, y in tqdm(train_dataloader, desc=f"[Fold {fold_idx+1}] Train Epoch {epoch}", leave=False):
            imgs, y = imgs.to(device), y.to(device)
            
            preds, loss = model(imgs, y)
            
            # L1 regularization
            l1_lambda = 1e-7
            reg_loss = sum(param.abs().sum() for param in model.parameters())
            loss = loss + l1_lambda * reg_loss
            
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_r2_scores.append(weighted_r2(y, preds, weights).item())
            
            # Track individual R2 scores
            r2_dict = weighted_r2_single(y, preds)
            for target_name, r2_value in r2_dict.items():
                train_r2_individual[target_name].append(r2_value)
        
        avg_train_loss = train_loss / len(train_dataloader)
        avg_train_r2 = sum(train_r2_scores) / len(train_r2_scores)
        avg_train_r2_individual = {k: sum(v) / len(v) for k, v in train_r2_individual.items()}
        
        # VALIDATION
        model.eval()
        val_loss = 0
        val_r2_scores = []
        val_r2_individual = {
            "Dry_Green_g": [],
            "Dry_Dead_g": [],
            "Dry_Clover_g": [],
            "GDM_g": [],
            "Dry_Total_g": []
        }
        
        with torch.no_grad():
            for imgs, _, y in tqdm(val_dataloader, desc=f"[Fold {fold_idx+1}] Val Epoch {epoch}", leave=False):
                imgs, y = imgs.to(device), y.to(device)
                preds, loss = model(imgs, y)
                val_loss += loss.item()
                val_r2_scores.append(weighted_r2(y, preds, weights).item())
                
                # Track individual R2 scores
                r2_dict = weighted_r2_single(y, preds)
                for target_name, r2_value in r2_dict.items():
                    val_r2_individual[target_name].append(r2_value)
        
        avg_val_loss = val_loss / len(val_dataloader)
        avg_val_r2 = sum(val_r2_scores) / len(val_r2_scores)
        avg_val_r2_individual = {k: sum(v) / len(v) for k, v in val_r2_individual.items()}
        
        val_losses.append(avg_val_loss)
        train_losses.append(avg_train_loss)
        val_r2_history.append(avg_val_r2)
        train_r2_history.append(avg_train_r2)
        
        epoch_time = time.time() - epoch_start_time
        
        # Save best model for this fold
        is_best = False
        if avg_val_r2 > best_val_r2:
            best_val_r2 = avg_val_r2
            best_epoch = epoch
            best_val_r2_individual = avg_val_r2_individual.copy()  # Save best per-target metrics
            best_model_path = f"{fold_dir}/best.pth"
            torch.save(model.state_dict(), best_model_path)
            is_best = True
            print(f"✓ New best model saved! Val R2: {best_val_r2:.4f} at epoch {epoch}")
        
        # Enhanced W&B logging with comprehensive metrics
        log_dict = {
            # Fold identification
            "fold": fold_idx + 1,
            "epoch": epoch,
            "epoch_time": epoch_time,
            
            # Overall metrics
            "train/loss": avg_train_loss,
            "train/r2_weighted": avg_train_r2,
            "val/loss": avg_val_loss,
            "val/r2_weighted": avg_val_r2,
            "val/best_r2": best_val_r2,
            
            # Train R2 per target
            "train/r2_dry_green": avg_train_r2_individual["Dry_Green_g"],
            "train/r2_dry_dead": avg_train_r2_individual["Dry_Dead_g"],
            "train/r2_dry_clover": avg_train_r2_individual["Dry_Clover_g"],
            "train/r2_gdm": avg_train_r2_individual["GDM_g"],
            "train/r2_dry_total": avg_train_r2_individual["Dry_Total_g"],
            
            # Val R2 per target
            "val/r2_dry_green": avg_val_r2_individual["Dry_Green_g"],
            "val/r2_dry_dead": avg_val_r2_individual["Dry_Dead_g"],
            "val/r2_dry_clover": avg_val_r2_individual["Dry_Clover_g"],
            "val/r2_gdm": avg_val_r2_individual["GDM_g"],
            "val/r2_dry_total": avg_val_r2_individual["Dry_Total_g"],
            
            # Overfitting indicators
            "metrics/train_val_loss_diff": avg_train_loss - avg_val_loss,
            "metrics/train_val_r2_diff": avg_train_r2 - avg_val_r2,
            "metrics/is_best_epoch": int(is_best),
            
            # Learning rate
            "optimizer/learning_rate": optimizer.param_groups[0]["lr"],
        }
        
        wandb.log(log_dict)
        
        # Print progress every 10 epochs
        if epoch % 10 == 0 or epoch == 1:
            print(f"\nEpoch {epoch}/{EPOCHS} | Time: {epoch_time:.2f}s")
            print(f"  Train Loss: {avg_train_loss:.4f} | Train R2: {avg_train_r2:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f} | Val R2: {avg_val_r2:.4f} | Best: {best_val_r2:.4f}")
            print(f"  Train R2 -> Green: {avg_train_r2_individual['Dry_Green_g']:.4f}, "
                  f"Dead: {avg_train_r2_individual['Dry_Dead_g']:.4f}, "
                  f"Clover: {avg_train_r2_individual['Dry_Clover_g']:.4f}, "
                  f"GDM: {avg_train_r2_individual['GDM_g']:.4f}, "
                  f"Total: {avg_train_r2_individual['Dry_Total_g']:.4f}")
            print(f"  Val R2   -> Green: {avg_val_r2_individual['Dry_Green_g']:.4f}, "
                  f"Dead: {avg_val_r2_individual['Dry_Dead_g']:.4f}, "
                  f"Clover: {avg_val_r2_individual['Dry_Clover_g']:.4f}, "
                  f"GDM: {avg_val_r2_individual['GDM_g']:.4f}, "
                  f"Total: {avg_val_r2_individual['Dry_Total_g']:.4f}")
    
    # Save last model for this fold
    last_model_path = f"{fold_dir}/last.pth"
    torch.save(model.state_dict(), last_model_path)
    
    fold_time = time.time() - fold_start_time
    
    # Store fold results for summary (using BEST epoch metrics, not last epoch!)
    fold_results = {
        "fold": fold_idx + 1,
        "best_val_r2": best_val_r2,
        "best_epoch": best_epoch,
        "final_train_loss": avg_train_loss,
        "final_val_loss": avg_val_loss,
        "final_train_r2": avg_train_r2,
        "final_val_r2": avg_val_r2,
        "fold_time": fold_time,
        "best_val_r2_per_target": best_val_r2_individual  # Use best metrics, not last!
    }
    all_folds_results.append(fold_results)
    
    # Log fold summary to W&B
    wandb.run.summary["fold"] = fold_idx + 1
    wandb.run.summary["best_val_r2"] = best_val_r2
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.run.summary["fold_time_hours"] = fold_time / 3600
    wandb.run.summary["final_train_r2"] = avg_train_r2
    wandb.run.summary["final_val_r2"] = avg_val_r2
    
    # Log best per-target metrics
    for target_name, r2_value in best_val_r2_individual.items():
        wandb.run.summary[f"best_val_r2_{target_name}"] = r2_value
    
    print(f"\n{'='*80}")
    print(f"Fold {fold_idx + 1} completed! Time: {fold_time/3600:.2f} hours")
    print(f"Best validation R2: {best_val_r2:.4f} achieved at epoch {best_epoch}")
    print(f"Best per-target R2:")
    for target_name, r2_value in best_val_r2_individual.items():
        print(f"  {target_name}: {r2_value:.4f}")
    print(f"Models saved in: {fold_dir}/")
    print(f"  - best.pth (epoch {best_epoch})")
    print(f"  - last.pth (epoch {EPOCHS})")
    print(f"{'='*80}\n")
    
    # Finish W&B run for this fold
    wandb.finish()

# Calculate and display cross-validation summary
print(f"\n{'='*80}")
print(f"K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*80}")

avg_best_r2 = sum([r["best_val_r2"] for r in all_folds_results]) / N_FOLDS
std_best_r2 = np.std([r["best_val_r2"] for r in all_folds_results])
avg_final_val_r2 = sum([r["final_val_r2"] for r in all_folds_results]) / N_FOLDS

print(f"\nOverall Performance:")
print(f"  Average Best Val R2: {avg_best_r2:.4f} ± {std_best_r2:.4f}")
print(f"  Average Final Val R2: {avg_final_val_r2:.4f}")

print(f"\nPer-Fold Results:")
for result in all_folds_results:
    print(f"  Fold {result['fold']}: Best R2 = {result['best_val_r2']:.4f} "
          f"(epoch {result['best_epoch']}) | Time: {result['fold_time']/3600:.2f}h")

print(f"\nPer-Target Average R2 (at best epochs across all folds):")
for target in ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]:
    avg_r2 = sum([r["best_val_r2_per_target"][target] for r in all_folds_results]) / N_FOLDS
    std_r2 = np.std([r["best_val_r2_per_target"][target] for r in all_folds_results])
    print(f"  {target}: {avg_r2:.4f} ± {std_r2:.4f}")

print(f"\n{'='*80}")
print(f"All {N_FOLDS} folds completed!")
print(f"Models saved in train_results/ directory")
print(f"Best model for each fold is at fold*/best.pth")
print(f"{'='*80}")



FOLD 1/5
Train batches: 35
Val batches: 9


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
                                                                                                                                                                 

✓ New best model saved! Val R2: -3.6364 at epoch 1

Epoch 1/1000 | Time: 12.93s
  Train Loss: 1.8481 | Train R2: -3.0084
  Val Loss: 1.6614 | Val R2: -3.6364 | Best: -3.6364
  Train R2 -> Green: -2.0698, Dead: -1.4282, Clover: -0.4857, GDM: -2.9137, Total: -4.0546
  Val R2   -> Green: -2.6926, Dead: -1.4147, Clover: -0.7489, GDM: -3.7123, Total: -4.8166


✓ New best model saved! Val R2: -3.5000 at epoch 2


...


Epoch 840/1000 | Time: 11.20s
  Train Loss: 0.4347 | Train R2: -0.9118
  Val Loss: 0.2406 | Val R2: 0.1628 | Best: 0.3321
  Train R2 -> Green: -0.7389, Dead: -0.3162, Clover: -0.2296, GDM: -1.0711, Total: -1.1383
  Val R2   -> Green: 0.3581, Dead: -0.0700, Clover: 0.1071, GDM: 0.2583, Total: 0.1433


✓ New best model saved! Val R2: 0.3405 at epoch 848



Epoch 850/1000 | Time: 12.36s
  Train Loss: 0.4172 | Train R2: -0.0120
  Val Loss: 0.2507 | Val R2: 0.1798 | Best: 0.3405
  Train R2 -> Green: 0.0010, Dead: -0.0357, Clover: 0.3328, GDM: -0.1346, Total: -0.0298
  Val R2   -> Green: 0.4193, Dead: -0.0426, Clover: -0.1522, GDM: 0.4074, Total: 0.1518



Epoch 860/1000 | Time: 11.92s
  Train Loss: 0.4133 | Train R2: -0.0585
  Val Loss: 0.2395 | Val R2: 0.2305 | Best: 0.3405
  Train R2 -> Green: 0.0674, Dead: -0.0980, Clover: -0.0554, GDM: 0.0284, Total: -0.1111
  Val R2   -> Green: 0.3739, Dead: 0.0658, Clover: -0.0319, GDM: 0.3381, Total: 0.2442


                                                                                                                                                                 
KeyboardInterrupt



In [15]:
import transformers
transformers.__version__

'4.57.3'

In [16]:
wandb.finish()

epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇██
epoch_time,▂▄▁▄▄▃▂▁▁▁▁▁▄▁▄▄▄▄▅▁▄▁▁▂▄▁▂▁▄▁▁▃▅▇▄▄█▂▃▃
fold,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/is_best_epoch,▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/train_val_loss_diff,▁▃█▆▆▇▇▆▅▆▅▆▆▅▅▅▅▇▅▄▅▅▅▄▄▅▅▅▆▅▅▄▄▅▄▅▆▅▅▄
metrics/train_val_r2_diff,▂▂▂▁▆▇▅▄▄▄▅▆▂▆▅▆▁▆▄▃▇▁▆▇▃▁█▃▄▄▂▅▄▆▆▇▄▄▆▇
optimizer/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▆▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/r2_dry_clover,▆▂▄▃▅▅▄▃▆▅▅▄▇▄▄▆▇▄▅▁▂▆▆▄▇▅▆▇▅▆▆▆▇▆▂▄▅█▅▅
train/r2_dry_dead,▂▄▅▇▇▄▅▆█▆█▆▂▄▅▆▇▇█▅▄▅█▄▁▂▇▆█▄▆▆▇▅▅▇▄▆█▇
+12,...


In [ ]:
torch.save(model.state_dict(), "image2biomass_weights_last.pth")
print("Model saved to image2biomass_weights_submission.pth")

In [ ]:
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

model = Image2BiomassModel().to(device)
model.load_state_dict(torch.load("image2biomass_weights_resnet50.pth", map_location=device))
model.eval()

rows = []

target_cols = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]

test_dataset = Image2BioMassTestDataset(
    dataset_path="/kaggle/input/csiro-biomass/",
    img_transform=val_transform
    # img_transform=train_transform,
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

with torch.no_grad():
    for imgs, _, sample_ids in tqdm(test_dataloader, desc="Inference"):
        imgs = imgs.to(device)

        # (B, 3) - model outputs: [Dry_Green_g, Dry_Dead_g, Dry_Clover_g]
        y_pred, _ = model(imgs, y=None)
        print("y_pred transformed:", y_pred)

        y_pred = target_untransform(y_pred).cpu().numpy()
        
        print("y_pred pure:", y_pred)
        # extract 3 predictions in the correct order
        dg = y_pred[:, 0]  # Dry_Green_g
        dd = y_pred[:, 1]  # Dry_Dead_g
        dc = y_pred[:, 2]  # Dry_Clover_g

        # compute extra targets
        gdm = dg + dc
        dry_total = dg + dd + dc

        preds5 = np.stack([dg, dd, dc, gdm, dry_total], axis=1)
        np.set_printoptions(suppress=True, precision=4)
        # print(preds5)

        # build submission rows
        for sid, pred_vec in zip(sample_ids, preds5):
            for col, value in zip(target_cols, pred_vec):
                rows.append({
                    "sample_id": f"{sid}__{col}",
                    "target": float(value)
                })

df_submit = pd.DataFrame(rows)
df_submit.to_csv("submission.csv", index=False)
print("Saved submission.csv")
df_submit.head(20)

In [ ]:
import shutil

shutil.make_archive("model_weights", "zip", "/kaggle/working", "image2biomass_weights_resnet50.pth")
from IPython.display import FileLink
FileLink("model_weights.zip")

In [ ]:
# target_untransform(3.8318)

# K-Fold Results Summary

In [ ]:
import os
import glob
import pandas as pd

# List all saved models
print("="*80)
print("SAVED MODELS IN train_results/")
print("="*80)

for i in range(N_FOLDS):
    fold_dir = f"train_results/fold{i+1}"
    if os.path.exists(fold_dir):
        print(f"\nFold {i+1}: ({fold_dir}/)")
        
        best_path = os.path.join(fold_dir, "best.pth")
        last_path = os.path.join(fold_dir, "last.pth")
        
        if os.path.exists(best_path):
            size_mb = os.path.getsize(best_path) / (1024**2)
            print(f"  ✓ best.pth ({size_mb:.2f} MB)")
        
        if os.path.exists(last_path):
            size_mb = os.path.getsize(last_path) / (1024**2)
            print(f"  ✓ last.pth ({size_mb:.2f} MB)")

print("\n" + "="*80)

# Count total models
total_models = len(glob.glob("train_results/*/best.pth")) + len(glob.glob("train_results/*/last.pth"))
print(f"Total models saved: {total_models}")

# Create summary table
if 'all_folds_results' in globals():
    print("\n" + "="*80)
    print("CROSS-VALIDATION RESULTS TABLE")
    print("="*80)
    
    df_results = pd.DataFrame(all_folds_results)
    
    # Basic stats
    print(f"\n{df_results[['fold', 'best_val_r2', 'best_epoch', 'final_val_r2']].to_string(index=False)}")
    
    # Calculate statistics
    print(f"\n{'='*80}")
    print(f"STATISTICS ACROSS FOLDS")
    print(f"{'='*80}")
    print(f"Best Val R2:")
    print(f"  Mean: {df_results['best_val_r2'].mean():.4f}")
    print(f"  Std:  {df_results['best_val_r2'].std():.4f}")
    print(f"  Min:  {df_results['best_val_r2'].min():.4f} (Fold {df_results.loc[df_results['best_val_r2'].idxmin(), 'fold']:.0f})")
    print(f"  Max:  {df_results['best_val_r2'].max():.4f} (Fold {df_results.loc[df_results['best_val_r2'].idxmax(), 'fold']:.0f})")
    
    print(f"\nFinal Val R2:")
    print(f"  Mean: {df_results['final_val_r2'].mean():.4f}")
    print(f"  Std:  {df_results['final_val_r2'].std():.4f}")
    
    print(f"\nTraining Time:")
    print(f"  Total: {df_results['fold_time'].sum()/3600:.2f} hours")
    print(f"  Per Fold: {df_results['fold_time'].mean()/3600:.2f} hours (avg)")

print("="*80)